# 3D Gaussian Splatting v2 — DJI Avata 360

Pre-computed COLMAP (local, 1h) → Train Gaussian Splatting on Colab A100.

**Upload to Drive first:** `DroneCV/gaussian_splat_data/images_v3.zip` + `colmap_output.zip`

In [ ]:
import torch, os, subprocess, shutil, zipfile, glob
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

assert torch.cuda.is_available(), 'GPU required!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Ready')

In [ ]:
# Prepare data directory in gaussian-splatting expected format
DRIVE = '/content/drive/MyDrive/DroneCV/gaussian_splat_data'
DATA_DIR = '/content/data'
os.makedirs(f'{DATA_DIR}/sparse/0', exist_ok=True)

# Extract images
print('Extracting images...')
with zipfile.ZipFile(f'{DRIVE}/images_v3.zip', 'r') as z:
    for m in tqdm(z.namelist(), desc='Images'):
        z.extract(m, DATA_DIR)
print(f'  {len(glob.glob(f"{DATA_DIR}/images/*.jpg"))} images')

# Extract COLMAP sparse model
print('Extracting COLMAP...')
with zipfile.ZipFile(f'{DRIVE}/colmap_output.zip', 'r') as z:
    z.extractall('/content/colmap_tmp')

# Copy to expected location
for f in glob.glob('/content/colmap_tmp/colmap/sparse/1/*'):
    shutil.copy(f, f'{DATA_DIR}/sparse/0/')

print(f'  COLMAP files: {os.listdir(f"{DATA_DIR}/sparse/0")}')
print('\n✅ Data ready')

In [ ]:
# Install gaussian-splatting
%cd /content
!rm -rf gaussian-splatting
!git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive 2>&1 | tail -2
%cd /content/gaussian-splatting
!pip install -q plyfile tqdm
!pip install -q submodules/diff-gaussian-rasterization 2>&1 | tail -1
!pip install -q submodules/simple-knn 2>&1 | tail -1
!pip install -q submodules/fused-ssim 2>&1 | tail -1
print('\n✅ Gaussian Splatting installed')

In [ ]:
# Train!
%cd /content/gaussian-splatting

ITERS = 7000
MODEL_PATH = '/content/output'

!python train.py \
  -s /content/data \
  --iterations {ITERS} \
  --model_path {MODEL_PATH} \
  --sh_degree 3

# Verify output exists
if os.path.exists(f'{MODEL_PATH}/cfg_args'):
    print(f'\n✅ Training complete! Model at {MODEL_PATH}')
    !ls {MODEL_PATH}/
else:
    print('\n⚠️ cfg_args not found, checking...')
    !find /content/output -type f | head -20

In [ ]:
# Render novel views
%cd /content/gaussian-splatting

!python render.py \
  -s /content/data \
  --model_path /content/output \
  --skip_test

# Display renders
import glob
renders = sorted(glob.glob('/content/output/train/ours_*/renders/*.png'))[:12]
if renders:
    fig, axes = plt.subplots(2, min(6, len(renders)//2+1), figsize=(18, 6))
    axes = np.array(axes).flatten()
    for i, r in enumerate(renders[:len(axes)]):
        axes[i].imshow(plt.imread(r)); axes[i].axis('off')
    plt.suptitle(f'Gaussian Splatting — Avata 360 over Jorvas ({ITERS} iter)', fontsize=13)
    plt.tight_layout()
    plt.savefig('/content/gs_renders.png', dpi=150)
    plt.show()
    print(f'\n✅ {len(renders)} renders saved')
else:
    print('No renders found. Showing point cloud instead...')
    ply_files = glob.glob('/content/output/point_cloud/*/point_cloud.ply')
    if ply_files:
        print(f'Model saved at: {ply_files[0]}')
        print(f'Size: {os.path.getsize(ply_files[0])/1e6:.1f} MB')

In [ ]:
# Save to Drive
SAVE_DIR = f'{DRIVE}/output'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy renders
if os.path.exists('/content/gs_renders.png'):
    shutil.copy('/content/gs_renders.png', f'{SAVE_DIR}/gs_renders.png')

# Copy model
ply_files = glob.glob('/content/output/point_cloud/*/point_cloud.ply')
if ply_files:
    shutil.copy(ply_files[0], f'{SAVE_DIR}/point_cloud.ply')
    print(f'Saved model: {os.path.getsize(ply_files[0])/1e6:.1f} MB')

# Copy a few individual renders
for r in renders[:6]:
    shutil.copy(r, f'{SAVE_DIR}/{os.path.basename(r)}')

print(f'\n✅ Saved to {SAVE_DIR}')

## Done!

| Step | Where | Result |
|------|-------|--------|
| Perspective extraction | Local | 432 ground-facing images from 8K equirect |
| COLMAP SfM | Local (1h CPU) | 386/432 registered, 50,958 3D points |
| Gaussian Splatting | Colab A100 | Photorealistic 3D model |

**Next:** Visual localization against this model for cm-level GNSS-denied navigation.